# Mastery Workbook: 29 to 31 Oct 2026

115 exercises on everything from the last three days of work: files and paths, reading types correctly,
strings and regex, combining and cleaning, lookups and joins, group calculations, dates, checks and output,
Polars, SQL, and five capstones that rebuild real reports.

**How it works**

1. Keep these files in one folder: `Mastery_Workbook.ipynb`, `practice_data.py`, `checker.py`, `solutions.py`.
2. Install once: `pip install pandas openpyxl xlsxwriter polars pyarrow duckdb`.
3. Run the **Setup** cell. The first run writes practice files to a `practice_data` folder (Stringee and Uno exports,
   a ledger, team lists, transactions, and the leads/call tables). They copy the shape of the real files with made-up data.
4. Each module starts with **notes and a worked example**. Run the example, change it, break it. Then do the exercises.
5. For each exercise, write your code, put the result in `ans` (or define the function asked for), and run `check(n, ans)`.
   You'll see `PASS`, or a hint about what's different. `progress()` shows your score by module.
6. `Mastery_Solutions.ipynb` has one worked answer per exercise. Open it only after you've tried, and when you do,
   close it and write the answer again from memory.

**Rules that make this work:** no copying from old notebooks, no asking Claude. If you're stuck for 15 minutes,
read that module's notes again, then the pandas docs, then the solution.

To start again: delete `practice_data/progress.json`. To rebuild the files: `build(force=True)`.

## Setup

In [ ]:
import os
import re
from pathlib import Path

import numpy as np
import pandas as pd

from practice_data import load, sql, build
from checker import check, progress

D = load()               # first run builds the practice files (a few seconds)
P = D['paths']           # every folder and file: P['stringee_dir'], P['ledger_csv'], ...
S = D['strings']         # text practice table for module 3
pd.set_option('display.max_columns', None)
print('Practice files in:', P['root'])
print('Tables in D:', [k for k, v in D.items() if isinstance(v, pd.DataFrame)])

In [ ]:
progress()

## Module 1. Files and paths

Every report starts by finding the right files. Two of this week's longest problems (the ledger that "stopped at the
27th" and the dialer report with "only the 28th") were input problems, not code problems.

| Task | Code |
| --- | --- |
| A folder or file path | `Path(r"C:\...\Stringee")`, then `folder / "file.xlsx"` |
| Matching files | `sorted(folder.glob("*.xlsx"))` |
| Everything in a folder | `folder.iterdir()` |
| Parts of a path | `p.name`, `p.stem`, `p.suffix`, `p.parent.name` |
| Last modified | `p.stat().st_mtime` (seconds since 1970) |
| Newest file | `max(files, key=lambda f: f.stat().st_mtime)` |
| Make a folder | `out_dir.mkdir(parents=True, exist_ok=True)` |

**Rules**

- Skip Excel lock files: `if not f.name.startswith("~$")`. Excel creates them while a workbook is open.
- Use full paths, not `os.chdir()`; changing the working folder silently affects every later cell.
- A file-count check should **list the files it found** in its error message.
- Before debugging any result, print which files were read, from which folder, with row counts and date ranges.

In [ ]:
folder = P['stringee_dir']
print('folder:', folder.name, '| inside:', folder.parent.name)

for f in sorted(folder.iterdir()):
    print(f"{f.name:32} suffix={f.suffix:6} size={f.stat().st_size:>7,} bytes")

exports = sorted(f for f in folder.glob('*.xlsx') if not f.name.startswith('~$'))
print('\nexports:', [f.name for f in exports])

t = pd.Timestamp('2026-10-28 14:05')
print('a dated name:', f"st_daily_{t:%Y-%m-%d}.xlsx", '| a path:', P['out_dir'] / 'example.csv')

### Files and paths: exercises

### Exercise 1. List the Stringee exports

`ans` = a **sorted list of file names** of every `.xlsx` in `P['stringee_dir']`,
skipping Excel lock files (names starting with `~$`).

In [ ]:
ans = None   # replace None with your answer

check(1, ans)

### Exercise 2. List the Uno exports

`ans` = a sorted list of the `.csv` file names in `P['uno_dir']`.

In [ ]:
ans = None   # replace None with your answer

check(2, ans)

### Exercise 3. A file check that explains itself

Write `need_files(folder, pattern, n)`: return the **sorted list of matching paths**
(skip `~$` files). If the count isn't `n`, raise `ValueError` with a message that lists the names found.

In [ ]:
def need_files(folder, pattern, n):
    ...   # your code

check(3, need_files)

### Exercise 4. The newest team list

`ans` = the **name** of the most recently modified `.csv` or `.xlsx` in `P['team_dir']`
(use `f.stat().st_mtime`).

In [ ]:
ans = None   # replace None with your answer

check(4, ans)

### Exercise 5. File names with a date range

Write `date_range_name(prefix, dates, ext)` where `dates` is a datetime Series.
One calendar date → `'Summary_2026-10-28.csv'`; several → `'Summary_2026-10-28_to_2026-10-30.csv'`.

In [ ]:
def date_range_name(prefix, dates, ext):
    ...   # your code

check(5, date_range_name)

### Exercise 6. Read whatever format the file is

Write `read_any(path)`: `pd.read_csv` for `.csv`, `pd.read_excel` for `.xlsx`,
and raise `ValueError` for anything else.

In [ ]:
def read_any(path):
    ...   # your code

check(6, read_any)

### Exercise 7. Pieces of a path

For `p = P['ledger_csv']`, `ans` = a tuple `(parent folder name, file stem, suffix)`.

In [ ]:
ans = None   # replace None with your answer

check(7, ans)

### Exercise 8. Name the processed file

Combine every Stringee export, parse `Start time`, and build the processed file name with
your `date_range_name` (prefix `'st_daily'`, ext `'.xlsx'`). `ans` = the name.

In [ ]:
ans = None   # replace None with your answer

check(8, ans)

### Exercise 9. Rows per export

`ans` = a dict `{file name: number of rows}` for every Stringee export.

In [ ]:
ans = None   # replace None with your answer

check(9, ans)

### Exercise 10. Date range per export

`ans` = a dict `{file name: ('first date', 'last date')}` using each export's `Start time`,
dates as `'YYYY-MM-DD'` strings. (This is the check that found the wrong-folder problem.)

In [ ]:
ans = None   # replace None with your answer

check(10, ans)

## Module 2. Reading and types

Decide types **at the boundary**: IDs as text while reading, dates and durations converted right after, with explicit
formats and a check that stops the run if anything fails.

| Column kind | Problem if you do nothing | Fix |
| --- | --- | --- |
| Phone numbers, IDs | One blank cell turns the column into floats: `9.19e+11`, later `'919...0'` | `dtype={'Customer number': str}` when reading |
| Dates as text | Guessing can swap day and month (`11/03` → 11 March) | `pd.to_datetime(s, format=..., errors='coerce')` |
| Durations | Text or Excel times can't be summed | `pd.to_timedelta(s.astype(str), errors='coerce')` |
| Whole numbers with blanks | Become floats like `41672978.0` | `.astype('Int64')` (capital I) |
| Mostly empty columns | Read as a mix of floats (blank) and text | `dtype={'expiryDate': str}` |

| Format code | Means | Example |
| --- | --- | --- |
| `%Y-%m-%d` | 4-digit year, month, day | `2026-10-28` |
| `%m/%d/%Y` | month first | `10/28/2026` |
| `%d/%b/%Y` | day, short month name | `28/Oct/2026` |
| `%H:%M:%S` | 24-hour clock | `21:34:25` |
| `%I:%M:%S %p` | 12-hour clock + AM/PM | `09:34:25 PM` |
| `format="ISO8601"` | any ISO layout, with or without fractions | `2026-10-28 21:34:25.181` |

`errors="coerce"` turns failures into `NaT`; always follow it with a count of `NaT` that raises an error.

In [ ]:
phones = pd.Series([919225131053, None, 917000691441])
print('one blank makes floats:', phones.dtype, '->', phones.astype(str).tolist())

text = pd.Series(['10/28/2026 09:34:25 PM', '11/03/2026 10:15:00 AM', 'bad value'])
parsed = pd.to_datetime(text, format='%m/%d/%Y %I:%M:%S %p', errors='coerce')
print('\nparsed:', parsed.tolist())
print('unreadable:', parsed.isna().sum())

dur = pd.to_timedelta(pd.Series(['00:05:40', '00:00:08', None]), errors='coerce')
print('\ndurations:', dur.tolist(), '| total seconds:', dur.dt.total_seconds().sum())

ids = pd.Series([41672978.0, None, 41672977.0])
print('\nInt64:', ids.astype('Int64').tolist())

### Reading and types: exercises

### Exercise 11. Phone numbers as text

Read `stringee_2026-10-28_a.xlsx` with `dtype={'Customer number': str}`.
`ans` = the first 3 values of `Customer number` as a list.

In [ ]:
ans = None   # replace None with your answer

check(11, ans)

### Exercise 12. Spot the float damage

Read the same file **without** `dtype`, then `.astype(str)` the `Customer number` column.
`ans` = how many values end with `'.0'` (blanks forced the column to floats).

In [ ]:
ans = None   # replace None with your answer

check(12, ans)

### Exercise 13. Month-first text dates

`P['us_times_csv']` has `Start time` like `'10/29/2026 09:34:25 PM'` and two unreadable values.
Parse with an explicit format and `errors='coerce'`. `ans` = the earliest timestamp.

In [ ]:
ans = None   # replace None with your answer

check(13, ans)

### Exercise 14. Count what failed

Same file and parsing. `ans` = how many values became `NaT`.

In [ ]:
ans = None   # replace None with your answer

check(14, ans)

### Exercise 15. Uno start times

Read the two daily Uno CSVs (`uno_2026-10-28.csv`, `uno_2026-10-29.csv`). Build each start time from
`Call Date` (`'28/Oct/2026'`) and `Call Time` (`'11:58:14 pm'`). `ans` = the latest start time.

In [ ]:
ans = None   # replace None with your answer

check(15, ans)

### Exercise 16. Total talk time

Combine every Stringee export (all rows), convert `Answer duration` to timedelta.
`ans` = the total in whole seconds (an int).

In [ ]:
ans = None   # replace None with your answer

check(16, ans)

### Exercise 17. A forgiving duration parser

Write `to_td(s)` for a Series of text: `'HH:MM:SS'` (spaces allowed around it), whole seconds like
`'75'`, or blanks. Return timedeltas, with blanks as `NaT`.

In [ ]:
def to_td(s):
    ...   # your code

check(17, to_td)

### Exercise 18. Caller numbers as text

Read `uno_2026-10-28.csv` with `Caller Number` as text. `ans` = the number of distinct caller numbers.

In [ ]:
ans = None   # replace None with your answer

check(18, ans)

### Exercise 19. See a mixed column

Read `tx_1.csv` with default settings. `ans` = a dict `{type name: count}` for the values in `expiryDate`,
using `.map(lambda v: type(v).__name__).value_counts()`.

In [ ]:
ans = None   # replace None with your answer

check(19, ans)

### Exercise 20. Whole numbers with blanks

`ans` = `D['ark']['Ticket Id']` converted to pandas' nullable integer type `Int64`
(blanks stay blank; no `.0`).

In [ ]:
ans = None   # replace None with your answer

check(20, ans)

### Exercise 21. Read only what you need

Read `tx_1.csv` with `usecols=['ticketId', 'time', 'accountId']`. `ans` = the list of columns you got.

In [ ]:
ans = None   # replace None with your answer

check(21, ans)

### Exercise 22. How big is it?

Read all of `tx_1.csv`. `ans` = its memory use in MB from `memory_usage(deep=True).sum() / 1e6`,
rounded to 2 decimals.

In [ ]:
ans = None   # replace None with your answer

check(22, ans)

## Module 3. Strings and regex

`.str` applies a string method to every value of a column. Regular expressions find a pattern wherever it sits, which
is safer than cutting by position when prefixes vary (`S75577` vs `BR575346`).

| Pattern | Means |
| --- | --- |
| `\d+` | one or more digits |
| `\s*` | any number of spaces |
| `( ... )` | the part `str.extract` returns |
| `\b` | a word boundary: `\bagent_id` won't match `sub_agent_id` |
| `[^a-z0-9]` | any character that is NOT a lowercase letter or digit |
| `^` / `$` | start / end of the text |

| Method | Use |
| --- | --- |
| `.str.split('|').str[1]` | item 1 of each split (not `[1]`, which takes row 1) |
| `.str.extract(r'customer\s*:\s*(\d+)', expand=False)` | one captured group as a Series |
| `.str.replace(r':\s*\*+$', '', regex=True)` | remove a pattern |
| `.str.fullmatch(p)` | the WHOLE value matches (`contains` = anywhere, `match` = at the start) |
| `.str[-10:]` | last 10 characters |
| `s.where(cond)` | keep values where `cond` is True, blank elsewhere |

In [ ]:
info = pd.Series(['userId : 30628867 | customer : 919810007919 | agent_id : 8943 | agentType : agent_id'])
print('customer:', info.str.extract(r'customer\s*:\s*(\d+)', expand=False).iloc[0])
print('agent   :', info.str.extract(r'\bagent_id\s*:\s*(\d+)', expand=False).iloc[0])

ids = pd.Series(['917557160434 | 919810007919', '917557160434 | 916269785548'])
print('\n[1] takes a ROW :', ids.str.split('|')[1])
print('.str[1] per row  :', ids.str.split('|').str[1].str.strip().tolist())

print('\nnormalised:', re.sub(r'[^a-z0-9]', '', 'Did not Connected / Out of Service'.lower()))
times = pd.Series(['2026-10-01 00:59:48', '59:48.4'])
print('damaged?  :', times.str.fullmatch(r'\d{1,2}:\d{2}(\.\d+)?').tolist())

### Strings and regex: exercises

### Exercise 23. The second caller ID

`S = D['strings']`. `ans` = a Series: the part after `|` in `S['caller_ids']`, with spaces stripped.

In [ ]:
ans = None   # replace None with your answer

check(23, ans)

### Exercise 24. Customer from Contact Info

`ans` = a Series of the digits after `customer :` in `S['contact_info']`.

In [ ]:
ans = None   # replace None with your answer

check(24, ans)

### Exercise 25. Agent id, not agentType

`ans` = a Series of the digits after `agent_id :` (the `agentType : agent_id` part must not match).

In [ ]:
ans = None   # replace None with your answer

check(25, ans)

### Exercise 26. Drop the country code

`ans` = the customer numbers from exercise 24, cut to their last 10 digits.

In [ ]:
ans = None   # replace None with your answer

check(26, ans)

### Exercise 27. Digits out of an ID

`ans` = a Series of the digits in `S['br_id']`, e.g. `'BR575346: ****'` → `'575346'`.

In [ ]:
ans = None   # replace None with your answer

check(27, ans)

### Exercise 28. Remove a masked suffix

`ans` = `S['account_id']` without the trailing `': ****'`, e.g. `'CO657078: ****'` → `'CO657078'`.

In [ ]:
ans = None   # replace None with your answer

check(28, ans)

### Exercise 29. Normalise a header

Write `norm(name)`: lowercase, then keep only letters and digits.
`'Ticket Id'` → `'ticketid'`, `'Total PnL'` → `'totalpnl'`, `'Amount ($)'` → `'amount'`.

In [ ]:
def norm(name):
    ...   # your code

check(29, norm)

### Exercise 30. Clean dialer names

Write `clean_dialer_name(s)` for a Series: remove `'(...)'`, everything from `@`, everything from `;`,
collapse repeated spaces, strip, lower-case. `'Riya.S  (Ext 22)'` → `'riya.s'`.

In [ ]:
def clean_dialer_name(s):
    ...   # your code

check(30, clean_dialer_name)

### Exercise 31. Find damaged times

`ans` = how many values in `S['damaged_time']` are only minutes:seconds, like `'59:48.4'`
(use `.str.fullmatch`).

In [ ]:
ans = None   # replace None with your answer

check(31, ans)

### Exercise 32. Blank them, then parse

Turn the damaged values into blanks with `.where`, then parse the rest with `format='ISO8601'`.
`ans` = the parsed Series.

In [ ]:
ans = None   # replace None with your answer

check(32, ans)

## Module 4. Combining and cleaning

Read each file into a list, then `pd.concat` once. Tag rows with their source. Remove duplicates **on purpose**, with a
named subset, and count what each step removed.

| Task | Code |
| --- | --- |
| Combine files | `pd.concat([read(f).assign(source_file=f.name) for f in files], ignore_index=True)` |
| Duplicate rows | `df.drop_duplicates(subset=[...], keep='first')` |
| Duplicate column names | `df.columns[df.columns.duplicated()]`; keep first: `df.loc[:, ~df.columns.duplicated()]` |
| Headers that differ in style | match on `norm(name)`; a small `MANUAL` dict for names that differ in words |
| Two names → one target | `pd.Series(mapping).duplicated(keep=False)` finds the clash |
| Same columns as another table | `df.reindex(columns=target_cols)` (adds missing ones as blank) |
| Drop a code, keep blanks | `df[~df[col].isin(codes)]` |
| Real agent only | `df[df[col].notna() & df[col].ne('---')]` |

In [ ]:
files = sorted(P['uno_dir'].glob('*.csv'))
uno = pd.concat([pd.read_csv(f, dtype={'Caller Number': str}).assign(source_file=f.name) for f in files],
                ignore_index=True)
print(uno.groupby('source_file').size())
print('rows:', len(uno), '| after removing repeated Call Ids:', len(uno.drop_duplicates(subset='Call Id')))

wide = pd.DataFrame([[1, 2, 3]], columns=['a', 'b', 'a'])
print('\nduplicate names:', wide.columns[wide.columns.duplicated()].tolist())
print('first copy only:', wide.loc[:, ~wide.columns.duplicated()].columns.tolist())

### Combining and cleaning: exercises

### Exercise 33. Combine with a source column

Combine every Stringee export and add a `source_file` column. `ans` = the shape `(rows, columns)`.

In [ ]:
ans = None   # replace None with your answer

check(33, ans)

### Exercise 34. Overlapping exports

On the combined Stringee rows, floor `Start time` to the second and drop duplicates on
`Account`, `Customer number`, `Start time`. `ans` = how many rows were removed.

In [ ]:
ans = None   # replace None with your answer

check(34, ans)

### Exercise 35. Find duplicate column names

`ans` = a sorted list of the column names that appear more than once in `D['dup_cols']`.

In [ ]:
ans = None   # replace None with your answer

check(35, ans)

### Exercise 36. Keep the first copy

`ans` = `D['dup_cols']` keeping only the first copy of each column name.

In [ ]:
ans = None   # replace None with your answer

check(36, ans)

### Exercise 37. Line up two sets of headers

Map each name in `D['ark_columns']` to one in `D['main_columns']`: match on `norm()`, plus
`MANUAL = {'Account': 'accountId', 'Open Com': 'openCommission', 'Close Com': 'closeCommission'}`.
`ans` = a dict `{ark name: main name}` for the columns that match.

In [ ]:
ans = None   # replace None with your answer

check(37, ans)

### Exercise 38. Catch clashes

`D['clash_columns']` are ark headers; map them to `D['main_columns']` the same way.
`ans` = a sorted list of the ark names that would end up with the **same** target name.

In [ ]:
ans = None   # replace None with your answer

check(38, ans)

### Exercise 39. Reindex to the target columns

Rename `D['ark']` with your exercise-37 mapping (use the real `D['ark'].columns`), then
`.reindex(columns=...)` to `D['main_columns']` without `'id'`. `ans` = the resulting DataFrame.

In [ ]:
ans = None   # replace None with your answer

check(39, ans)

### Exercise 40. Overlapping Uno exports

Combine all three Uno CSVs and drop duplicate `Call Id`s. `ans` = `(rows before, rows after)`.

In [ ]:
ans = None   # replace None with your answer

check(40, ans)

### Exercise 41. Drop one end code safely

On the combined Stringee rows, drop `CAN_NOT_MAKE_CALL` (rows with a blank code must stay).
`ans` = rows left.

In [ ]:
ans = None   # replace None with your answer

check(41, ans)

### Exercise 42. Calls that have an agent

On the combined Stringee rows, keep rows whose `Account` is not blank and not `'---'`. `ans` = rows left.

In [ ]:
ans = None   # replace None with your answer

check(42, ans)

## Module 5. Lookups and joins

Most wrong numbers this week came from joins: a "latest row" taken from the wrong sort, or a table with several rows
per key multiplying the other side.

| Task | Code |
| --- | --- |
| Latest row per key | `df.sort_values('CreatedAt', ascending=False).drop_duplicates('ClientID')` |
| Same, another way | `df.loc[df.groupby('ClientID')['CreatedAt'].idxmax()]` |
| Latest value only | `df.groupby('ClientID', as_index=False)['CreatedAt'].max()` |
| Fill one column from a lookup | `key.map(lookup.set_index('ClientID')['Balance'])` |
| Group value on every row | `df.groupby('ClientID')['CreatedAt'].transform('max')` |
| Guard a merge | `pd.merge(..., validate='many_to_one')` raises `MergeError` on repeats |

**Fan-out:** merging on a key that repeats on the right side copies left rows. Always compare `len()` before and after
a merge, or `COUNT(*)` with `COUNT(DISTINCT key)` in SQL.

In [ ]:
led = D['ledger']
latest = led.sort_values('CreatedAt', ascending=False).drop_duplicates('ClientID')[['ClientID', 'Balance']]
print(latest.head(3))

ids = pd.Series([' C100 ', 'C102', 'C999'])
print('\nmapped:', ids.str.strip().map(latest.set_index('ClientID')['Balance']).tolist())

left = pd.DataFrame({'name': ['a', 'b']})
right = pd.DataFrame({'name': ['a', 'a', 'b'], 'crm': ['x1', 'x2', 'y']})
print('\nrows before:', len(left), '| after merge:', len(left.merge(right, on='name')))

### Lookups and joins: exercises

### Exercise 43. Latest balance per client

From `D['ledger']`: `ans` = a DataFrame `ClientID`, `Balance` with each client's balance on their latest
`CreatedAt` (sort by date, newest first, keep the first row per client).

In [ ]:
ans = None   # replace None with your answer

check(43, ans)

### Exercise 44. Same result with idxmax

Get the same answer using `D['ledger'].groupby('ClientID')['CreatedAt'].idxmax()` and `.loc`.
`ans` = DataFrame `ClientID`, `Balance`.

In [ ]:
ans = None   # replace None with your answer

check(44, ans)

### Exercise 45. Last trade date

`ans` = a DataFrame `ClientID`, `last_trade`: the latest `CreatedAt` among `BILL` rows.

In [ ]:
ans = None   # replace None with your answer

check(45, ans)

### Exercise 46. Fill a column with .map()

Strip spaces from `D['traded_ids']['Zynixx Id']`, then map each id to its latest balance.
`ans` = the Series of balances in the same order (NaN where not found).

In [ ]:
ans = None   # replace None with your answer

check(46, ans)

### Exercise 47. Who got no balance

`ans` = a sorted list of stripped ids from exercise 46 with no balance, excluding `'No New Id'`.

In [ ]:
ans = None   # replace None with your answer

check(47, ans)

### Exercise 48. Let validate catch repeats

Run `pd.merge(left, D['ledger'][['ClientID', 'Balance']], on='ClientID', validate='many_to_one')` where
`left = D['traded_ids'].assign(ClientID=D['traded_ids']['Zynixx Id'].str.strip())`, inside `try/except`.
`ans` = the exception's class name.

In [ ]:
ans = None   # replace None with your answer

check(48, ans)

### Exercise 49. Spot a fan-out

Clean `D['team_old']['Dialer Name']` with your `clean_dialer_name`, then left-merge
`D['calls'][['Dialer Name']]` with it on `Dialer Name`. `ans` = `(rows before, rows after)`.

In [ ]:
ans = None   # replace None with your answer

check(49, ans)

### Exercise 50. Names shared by two people

In `D['team_old']`: drop rows whose `Email` contains `'inactive'` (any case), clean the names,
drop duplicate (`Dialer Name`, `Email`). `ans` = a sorted list of names still linked to more than one email.

In [ ]:
ans = None   # replace None with your answer

check(50, ans)

### Exercise 51. Unmatched calls per source

Left-merge `D['calls_all']` (cleaned calls) with `D['team']` (cleaned team list) on `Dialer Name`.
`ans` = a dict `{Source: calls with no CRM ID}`.

In [ ]:
ans = None   # replace None with your answer

check(51, ans)

### Exercise 52. The group's value on every row

`ans` = a Series: for every ledger row, that client's latest `CreatedAt` (use `transform('max')`).

In [ ]:
ans = None   # replace None with your answer

check(52, ans)

## Module 6. Group calculations

Compute helper columns first (connected flag, seconds, masked talk time), then aggregate with plain `sum`, `count`,
`nunique`. That is faster and clearer than `lambda`s inside `.agg`.

| Task | Code |
| --- | --- |
| Several results per group | `.agg(dialed=('Call Status', 'count'), uniq=('Number', 'nunique'))` |
| Sum only some rows | `talk_s.where(connected, 0)`, then `sum` |
| Previous row in the group | `df.groupby(keys)[col].shift()` (sort first!) |
| Negative to zero | `gap.clip(lower=pd.Timedelta(0))` |
| Time window | `tod.between(pd.Timedelta('09:30:00'), pd.Timedelta('18:30:00'))` |
| Bands / tiers | `np.select([cond1, cond2], ['A', 'B'], default='C')` (first true wins) |
| Rows to columns | `pivot_table(index=..., columns=..., values=..., aggfunc='count', fill_value=0)` |
| Days, not rows | `aggfunc='nunique'` on `CreatedAt.dt.normalize()` |
| Month labels | `dt.to_period('M').astype(str)` → `'2026-10'` |

In [ ]:
calls = pd.DataFrame({
    'CRM ID': ['a', 'a', 'a', 'b'],
    'Start': pd.to_datetime(['2026-10-28 09:40:00', '2026-10-28 09:42:30', '2026-10-28 09:46:00', '2026-10-28 10:00:00']),
    'Dur': pd.to_timedelta(['00:02:00', '00:00:30', '00:05:00', '00:01:00'])})
calls = calls.sort_values(['CRM ID', 'Start'])
prev_start = calls.groupby('CRM ID')['Start'].shift()
prev_end = prev_start + calls.groupby('CRM ID')['Dur'].shift()
calls['gap'] = (calls['Start'] - prev_end).clip(lower=pd.Timedelta(0))
calls['flag'] = np.where(calls['gap'].dt.total_seconds() > 60, 'Yes', 'No')
print(calls)

### Group calculations: exercises

### Exercise 53. Calls per agent per day

From `D['calls']`: `ans` = a DataFrame `CRM ID`, `Date`, `calls`.

In [ ]:
ans = None   # replace None with your answer

check(53, ans)

### Exercise 54. Named aggregation

Per `CRM ID` and `Date`: `dialed` (rows), `connected` (Call Status == 'connected'),
`unique_numbers` (distinct Number). `ans` = a DataFrame with those 5 columns.

In [ ]:
ans = None   # replace None with your answer

check(54, ans)

### Exercise 55. Talk time of connected calls only

`ans` = a DataFrame `CRM ID`, `Date`, `talk_s`: whole seconds of `Talk Time`, counting connected calls
only (`.where(connected, 0)` before summing).

In [ ]:
ans = None   # replace None with your answer

check(55, ans)

### Exercise 56. Long connected calls

`ans` = a DataFrame `CRM ID`, `Date`, `gt30`: number of connected calls with talk time over 30 seconds.

In [ ]:
ans = None   # replace None with your answer

check(56, ans)

### Exercise 57. The previous call's start

`D['calls']` is sorted by Date, CRM ID, Call Start Time. `ans` = a Series of the previous call's start
for the same CRM ID and Date (`groupby(...).shift()`; NaT for each first call).

In [ ]:
ans = None   # replace None with your answer

check(57, ans)

### Exercise 58. Gap in seconds

gap = start − (previous start + previous `Total Call Duration`); negatives → 0, missing → 0.
`ans` = a Series of whole seconds (int).

In [ ]:
ans = None   # replace None with your answer

check(58, ans)

### Exercise 59. Flagged gaps

Count gaps over 60 seconds where **both** this call and the previous call start between 09:30 and 18:30
(inclusive, time of day). `ans` = an int.

In [ ]:
ans = None   # replace None with your answer

check(59, ans)

### Exercise 60. Break allowance

Sum the gap seconds per `CRM ID` and `Date`, then subtract 3600 where the total is over 3600.
`ans` = a DataFrame `CRM ID`, `Date`, `gap_s`.

In [ ]:
ans = None   # replace None with your answer

check(60, ans)

### Exercise 61. Login seconds

`login_s` = total call seconds (sum of whole-second `Total Call Duration`) + the adjusted gap from
exercise 60 + 3600. `ans` = a DataFrame `CRM ID`, `Date`, `login_s`.

In [ ]:
ans = None   # replace None with your answer

check(61, ans)

### Exercise 62. Attendance bands with np.select

From exercise 61: under 4h30 → `'Absent'`, under 6h → `'Half Day'`, under 8h30 → `'Warning'`,
otherwise `'Present'`. `ans` = a dict `{label: count}`.

In [ ]:
ans = None   # replace None with your answer

check(62, ans)

### Exercise 63. Hour slots

Label each call's start hour with `D['SLOT_LABELS']`: hour < 8 → `'0 to 8 AM'`, 8 → `'8 to 9 AM'`, …,
19 → `'19 to 20 PM'`, 20 or later → `'20+'` (hint: `(hour - 7).clip(0, 13)`). `ans` = a dict `{slot: calls}`.

In [ ]:
ans = None   # replace None with your answer

check(63, ans)

### Exercise 64. Calls per slot

`ans` = a pivot DataFrame (after `reset_index()`): columns `CRM ID`, `Date`, then one column per slot
that appears (calls, 0 if none).

In [ ]:
ans = None   # replace None with your answer

check(64, ans)

### Exercise 65. Trading days per month

From `BILL` rows in `D['ledger']`: `ans` = a pivot DataFrame (after `reset_index()`) with `ClientID` and one
column per month (`'2026-08'` text) holding the number of **distinct dates** traded.

In [ ]:
ans = None   # replace None with your answer

check(65, ans)

### Exercise 66. Check for the impossible

`ans` = the largest number in your exercise-65 month columns (it must be 31 or less).

In [ ]:
ans = None   # replace None with your answer

check(66, ans)

### Exercise 67. Total and average

From exercise 65, add `Total Trading Days` (sum of month columns) and `Average` (their mean, 2 decimals).
`ans` = a DataFrame `ClientID`, `Total Trading Days`, `Average`.

In [ ]:
ans = None   # replace None with your answer

check(67, ans)

### Exercise 68. Confidence tiers

`D['tiers']` has `last_txn`, `call_1_time` (latest missed call), `call_5_time` (first of the streak) and
`call_days`. Low if `call_days == 1` or `last_txn <= call_5_time`; else High if `last_txn > call_1_time`;
else Medium. `ans` = a dict `{tier: count}`.

In [ ]:
ans = None   # replace None with your answer

check(68, ans)

## Module 7. Dates and times

Use half-open ranges: `>= start` and `< end`. They catch every fraction of a second and never double-count midnight.

| Task | Code |
| --- | --- |
| Start of a day | `ts.normalize()` / `s.dt.normalize()` |
| Yesterday | `start = today - pd.Timedelta(days=1)`, rows with `start <= t < today` |
| Start of the month | `today.replace(day=1)` |
| Time of day | `t - t.dt.normalize()` (a timedelta to compare with 09:30) |
| UTC to Dubai | `.dt.tz_localize('UTC').dt.tz_convert('Asia/Dubai').dt.tz_localize(None)` |
| Every day in a range | `pd.date_range(first, last, freq='D')` |
| Days missing | `full_range.difference(pd.DatetimeIndex(days_seen))` |
| Date as text | `.dt.strftime('%Y-%m-%d')` |

In [ ]:
today = D['TODAY']
t = D['ledger']['CreatedAt']
yesterday = (t >= today - pd.Timedelta(days=1)) & (t < today)
print('today:', today.date(), '| rows yesterday:', yesterday.sum())

sample = t.head(3)
print('\nUTC  :', sample.tolist())
print('Dubai:', sample.dt.tz_localize('UTC').dt.tz_convert('Asia/Dubai').dt.tz_localize(None).tolist())
print('time of day:', (sample - sample.dt.normalize()).tolist())

### Dates and times: exercises

### Exercise 69. Yesterday

With `today = D['TODAY']`, `ans` = the number of ledger rows from yesterday (`>=` start, `<` end).

In [ ]:
ans = None   # replace None with your answer

check(69, ans)

### Exercise 70. The last 7 full days

`ans` = ledger rows from the 7 days before `D['TODAY']`, not including `D['TODAY']`.

In [ ]:
ans = None   # replace None with your answer

check(70, ans)

### Exercise 71. Month so far

`ans` = ledger rows from the 1st of `D['TODAY']`'s month up to, but not including, `D['TODAY']`.

In [ ]:
ans = None   # replace None with your answer

check(71, ans)

### Exercise 72. Days the calls were spread over

`D['last5']` lists each lead's last calls (`lead_id`, `rn`, `call_time`). `ans` = a DataFrame `lead_id`,
`call_days`: the number of distinct dates among `rn` 1 to 5.

In [ ]:
ans = None   # replace None with your answer

check(72, ans)

### Exercise 73. UTC to Dubai

Treat ledger `CreatedAt` as UTC, convert it to `'Asia/Dubai'` and drop the time zone.
`ans` = the converted Series.

In [ ]:
ans = None   # replace None with your answer

check(73, ans)

### Exercise 74. Inside working hours

`ans` = the number of calls in `D['calls']` starting between 09:30 and 18:30, inclusive.

In [ ]:
ans = None   # replace None with your answer

check(74, ans)

### Exercise 75. Calls per day

`ans` = a dict `{'YYYY-MM-DD': calls}` for `D['calls']`.

In [ ]:
ans = None   # replace None with your answer

check(75, ans)

### Exercise 76. Missing days

For the client `D['gap_client']`, look at `BILL` rows from 1 Oct 2026 onwards. `ans` = a sorted list of dates
(`'YYYY-MM-DD'`) between their first and last October BILL date on which they had **no** BILL
(hint: `pd.date_range(...).difference(...)`).

In [ ]:
ans = None   # replace None with your answer

check(76, ans)

### Exercise 77. Rows per month

`ans` = a dict `{'2026-08': rows, ...}` of ledger rows per month (`dt.to_period('M')`).

In [ ]:
ans = None   # replace None with your answer

check(77, ans)

### Exercise 78. Seconds to HH:MM:SS

Write `hms(seconds)` turning a Series of whole seconds into `'HH:MM:SS'` text (hours may go past 24).

In [ ]:
def hms(seconds):
    ...   # your code

check(78, hms)

## Module 8. Checks and output

A report should refuse to produce wrong numbers. Cheap checks catch most mistakes before anyone opens the file.

| Check | Code |
| --- | --- |
| Something impossible | `assert pivot.max().max() <= 31, "counting bills, not days"` |
| Unreadable values | `if s.isna().any(): raise ValueError(f"{s.isna().sum()} values unreadable")` |
| Two reports agree | merge their totals and count rows where they differ |
| What each step removed | `len()` before and after each filter, in a dict |
| Rows vs keys after a merge | `len(m)` vs `m[key].nunique()` |

**Excel output:** pandas writes a timedelta as a fraction of a day (`0.003935`). Write `seconds / 86400` and give the
column the `[h]:mm:ss` format. `ExcelWriter(path, mode='a', if_sheet_exists='replace')` adds or replaces one sheet.

**Secrets:** `os.environ['NAME']` reads a variable **by name**. Set it once with `setx NAME "value"` in a terminal, then
restart the editor. Never put a token's value in code.

In [ ]:
out = P['out_dir'] / 'example_durations.xlsx'
df = pd.DataFrame({'agent': ['a', 'b'], 'talk': pd.to_timedelta(['00:05:40', '01:02:03'])})
df['talk'] = df['talk'].dt.total_seconds() / 86_400
with pd.ExcelWriter(out, engine='xlsxwriter') as w:
    df.to_excel(w, index=False, sheet_name='CDR')
    w.sheets['CDR'].set_column(1, 1, 12, w.book.add_format({'num_format': '[h]:mm:ss'}))

from openpyxl import load_workbook
print('B2 format:', load_workbook(out)['CDR']['B2'].number_format)

def require_parsed(s, label):
    bad = int(s.isna().sum())
    if bad:
        raise ValueError(f"{bad} {label} values could not be read")
    return s

try:
    require_parsed(pd.Series([pd.Timestamp('2026-10-01'), pd.NaT]), 'start time')
except ValueError as e:
    print('stopped:', e)

### Checks and output: exercises

### Exercise 79. Two reports must agree

Count calls per `CRM ID` and `Date` two ways: `groupby().size()`, and the row sum of the slot columns of
your exercise-64 pivot. `ans` = how many rows differ (should be 0).

In [ ]:
ans = None   # replace None with your answer

check(79, ans)

### Exercise 80. A check that stops the run

Write `require_parsed(s, label)`: return `s` unchanged, or raise `ValueError` mentioning how many values are
missing (NaT/NaN).

In [ ]:
def require_parsed(s, label):
    ...   # your code

check(80, require_parsed)

### Exercise 81. Durations Excel can add up

Write `D['calls'][['CRM ID', 'Talk Time']].head(20)` to `P['out_dir'] / 'durations.xlsx'`, with
`Talk Time` as a fraction of a day (`seconds / 86400`) and the number format `[h]:mm:ss` on that column.
Read cell B2 back with `openpyxl.load_workbook`. `ans` = its `number_format`.

In [ ]:
ans = None   # replace None with your answer

check(81, ans)

### Exercise 82. Add a sheet to a workbook

Copy `team_2026-10-30.xlsx` to `P['out_dir'] / 'team_copy.xlsx'`, then add a sheet `result` with
`D['calls'].head()` using `ExcelWriter(..., mode='a', if_sheet_exists='replace')`. Run it twice.
`ans` = the workbook's sheet names (list).

In [ ]:
ans = None   # replace None with your answer

check(82, ans)

### Exercise 83. Excel's fraction of a day

`ans` = 5 minutes 40 seconds as a fraction of a day, rounded to 6 decimals.

In [ ]:
ans = None   # replace None with your answer

check(83, ans)

### Exercise 84. Rows versus distinct keys

Left-merge `D['customer_status_history']` onto `D['leads']` (`leads.customer_id` = `customer_id`).
`ans` = `(rows after the merge, distinct lead ids)`. Different numbers mean every lead repeats.

In [ ]:
ans = None   # replace None with your answer

check(84, ans)

### Exercise 85. Rows dropped at each step

From the combined Stringee rows: (1) drop `CAN_NOT_MAKE_CALL`; (2) floor `Start time` to the second and drop
duplicates on Account, Customer number, Start time; (3) drop blank or `'---'` Account.
`ans` = a dict `{'end_code': n1, 'duplicates': n2, 'no_agent': n3}` of rows removed at each step.

In [ ]:
ans = None   # replace None with your answer

check(85, ans)

### Exercise 86. Environment variables are read by name

In your cell, set `os.environ['PRACTICE_TOKEN'] = 'abc123'`, then read it back **by its name**.
`ans` = the length of the value you read.

In [ ]:
ans = None   # replace None with your answer

check(86, ans)

## Module 9. Polars (lazy)

A LazyFrame is a **plan**; nothing is read until `.collect()`. Cut the plan down first, collect last, and convert to
pandas only when the result is small.

| Call | Reads | Memory |
| --- | --- | --- |
| `pl.scan_csv(f)` | nothing yet | none |
| `lf.collect_schema().names()` | headers | tiny |
| `lf.select(pl.len()).collect()` | counts rows | tiny result |
| `lf.filter(...).select([...]).collect()` | only those rows and columns | what you kept |
| `lf.collect()` on everything | everything | all of it (this crashed the kernel) |
| `lf.sink_parquet(path)` | everything, in batches, to disk | small, steady |

`pl.concat(parts, how='diagonal_relaxed')` matches columns **by name** and widens types; different header styles
give separate half-empty columns. `pl.lit(value).alias(name)` adds the same value to every row.

In [ ]:
import polars as pl

files = sorted(P['tx_dir'].glob('*.csv'))
lf = pl.concat([pl.scan_csv(f, schema_overrides={'expiryDate': pl.String, 'comment': pl.String})
                for f in files], how='diagonal_relaxed')
print(type(lf).__name__, '->', lf.collect_schema().names()[:5], '...')

small = (lf.filter(pl.col('script') == 'TESLA')
           .group_by('action').agg(pl.len().alias('rows'), pl.col('amount').sum().alias('lots'))
           .collect())
print(small)

### Polars: exercises

### Exercise 87. A lazy plan

`lf = pl.concat([pl.scan_csv(f) for f in sorted(P['tx_dir'].glob('*.csv'))], how='diagonal_relaxed')`.
`ans` = the column names (list), without reading the data (`collect_schema()`).

In [ ]:
ans = None   # replace None with your answer

check(87, ans)

### Exercise 88. Count rows lazily

`ans` = the total number of rows, from `lf.select(pl.len()).collect().item()`.

In [ ]:
ans = None   # replace None with your answer

check(88, ans)

### Exercise 89. Filter before collecting

`ans` = the number of rows whose `accountId` is in `D['tx_accounts']` and `action == 'New'`,
using `.filter(...)` before `.collect()`.

In [ ]:
ans = None   # replace None with your answer

check(89, ans)

### Exercise 90. Group in Polars

`ans` = a dict `{script: rows}` using `lf.group_by('script').agg(pl.len())`.

In [ ]:
ans = None   # replace None with your answer

check(90, ans)

### Exercise 91. Tag each file

Scan each transactions CSV with `.with_columns(pl.lit(f.name).alias('source_file'))`, concat, then count.
`ans` = a dict `{file name: rows}`.

In [ ]:
ans = None   # replace None with your answer

check(91, ans)

### Exercise 92. When headers differ

Scan `P['tx_dir'] / 'tx_1.csv'` and `P['ark_dir'] / 'ark_1.csv'` and concat with `how='diagonal_relaxed'`.
`ans` = the number of columns (different names don't line up, so you get both sets).

In [ ]:
ans = None   # replace None with your answer

check(92, ans)

### Exercise 93. Sink to Parquet

Write `lf` to `P['out_dir'] / 'tx.parquet'` with `lf.sink_parquet(...)`, then count its rows with
`pl.scan_parquet(...).select(pl.len()).collect().item()`. `ans` = that count.

In [ ]:
ans = None   # replace None with your answer

check(93, ans)

### Exercise 94. Back to pandas, small

`ans` = a **pandas** DataFrame with `ticketId` and `amount` for `script == 'XAUUSD'`
(`select` + `filter`, `.collect()`, then `.to_pandas()`).

In [ ]:
ans = None   # replace None with your answer

check(94, ans)

## Module 10. SQL (PostgreSQL syntax, run with DuckDB)

`sql("...")` runs a query on the practice tables and returns a DataFrame. DuckDB follows PostgreSQL syntax for
everything used here (`ROW_NUMBER`, `DISTINCT ON`, `FILTER`, `BOOL_AND`, `STRING_AGG`, `ARRAY_AGG(...)[1]`,
`regexp_replace(..., 'g')`), so the same queries work in Metabase on PostgreSQL.

Tables: `leads(id, customer_id, tag)`, `call_logs(id, lead_id, started_at, status)`, `lead_calls(call_log_id,
disposition_id)`, `lead_call_dispositions(id, name)`, `customer_status_history(id, customer_id, previous_status,
new_status, occurred_at, created_at)`.

**Run order:** FROM/JOIN → WHERE → GROUP BY → HAVING → window functions → SELECT → DISTINCT → ORDER BY → LIMIT.
That's why a window result (`rn`) is filtered in the next CTE, never in the same SELECT.

**The pattern for "last N calls":** rank (`ROW_NUMBER() OVER (PARTITION BY lead_id ORDER BY started_at DESC, id DESC)`),
cut (`WHERE rn <= N`), group (`GROUP BY lead_id`), test (`HAVING COUNT(*) = N AND BOOL_AND(...)`).

In [ ]:
example = sql('''
    WITH ranked AS (
        SELECT cl.lead_id, cl.started_at, d.name AS disposition,
               ROW_NUMBER() OVER (PARTITION BY cl.lead_id ORDER BY cl.started_at DESC, cl.id DESC) AS rn
        FROM call_logs cl
        JOIN lead_calls lc ON lc.call_log_id = cl.id
        JOIN lead_call_dispositions d ON d.id = lc.disposition_id
        WHERE cl.status NOT IN ('placing', 'call_failed')
    )
    SELECT * FROM ranked WHERE lead_id = 4 ORDER BY rn
''')
example

### SQL: exercises

### Exercise 95. Newest call per lead

`ans = sql(...)`: `lead_id`, `started_at` of each lead's newest call, ignoring `status` `'placing'` and
`'call_failed'`, using `ROW_NUMBER()` (calls without a disposition count here).

In [ ]:
ans = None   # replace None with your answer

check(95, ans)

### Exercise 96. Last 3 all 'No Answer'

Rank each lead's **dispositioned** calls newest first (join `lead_calls` and `lead_call_dispositions`; skip
placing/call_failed). `ans` = a sorted list of lead ids whose last 3 are all `'No Answer'`
(`BOOL_AND` + `HAVING COUNT(*) = 3`).

In [ ]:
ans = None   # replace None with your answer

check(96, ans)

### Exercise 97. Same leads with FILTER

Same answer using `COUNT(*) FILTER (WHERE disposition = 'No Answer') = 3` instead of `BOOL_AND`.

In [ ]:
ans = None   # replace None with your answer

check(97, ans)

### Exercise 98. Latest disposition with DISTINCT ON

`ans` = `lead_id`, `disposition` of each lead's newest dispositioned call, using
`SELECT DISTINCT ON (lead_id) ... ORDER BY lead_id, started_at DESC, id DESC`.

In [ ]:
ans = None   # replace None with your answer

check(98, ans)

### Exercise 99. A window aggregate

On the ranked rows with `rn <= 5`, add `COUNT(*) OVER (PARTITION BY lead_id)`.
`ans` = the number of leads that have all 5.

In [ ]:
ans = None   # replace None with your answer

check(99, ans)

### Exercise 100. A sequence as text

`ans` = `lead_id`, `last_3`: the last 3 dispositions, newest first, joined with `' -> '`
(`STRING_AGG(... ORDER BY rn)`).

In [ ]:
ans = None   # replace None with your answer

check(100, ans)

### Exercise 101. Two-level breakdown

Over each lead's last 5 calls: count per disposition, then `STRING_AGG(disposition || ' x' || cnt, ', '
ORDER BY cnt DESC, disposition)`. `ans` = `lead_id`, `breakdown`, e.g. `'No Answer x3, Zero x2'`.

In [ ]:
ans = None   # replace None with your answer

check(101, ans)

### Exercise 102. Match names after normalising

Put the 8 names in `D['UNREACHABLE']` in a `VALUES` list. Normalise both sides with
`regexp_replace(lower(x), '[^a-z0-9]', '', 'g')`. `ans` = how many dispositioned calls are unreachable.

In [ ]:
ans = None   # replace None with your answer

check(102, ans)

### Exercise 103. Rows into columns

`ans` = `lead_id`, `call_1`, `call_2`, `call_3` (dispositions, newest first) using
`MAX(disposition) FILTER (WHERE rn = 1)` and so on.

In [ ]:
ans = None   # replace None with your answer

check(103, ans)

### Exercise 104. De-duplicate with DISTINCT ON

From `customer_status_history`, keep one row per (`customer_id`, `occurred_at`): the one with the latest
`created_at`. `ans` = the number of rows kept.

In [ ]:
ans = None   # replace None with your answer

check(104, ans)

### Exercise 105. Current status

On the de-duplicated rows: `ans` = `customer_id`, `rt_status` using
`(ARRAY_AGG(new_status ORDER BY occurred_at DESC))[1]`.

In [ ]:
ans = None   # replace None with your answer

check(105, ans)

### Exercise 106. Status journey

On the de-duplicated rows: `ans` = `customer_id`, `journey` = statuses oldest first joined with `' -> '`.

In [ ]:
ans = None   # replace None with your answer

check(106, ans)

### Exercise 107. Fan-out in SQL

`ans` = a tuple of two row counts: `leads LEFT JOIN customer_status_history` directly, and
`leads LEFT JOIN` a one-row-per-customer summary of it.

In [ ]:
ans = None   # replace None with your answer

check(107, ans)

### Exercise 108. A half-open date range

`ans` = the number of `call_logs` rows that started on 2026-10-29
(`started_at >= '2026-10-29' AND started_at < '2026-10-30'`).

In [ ]:
ans = None   # replace None with your answer

check(108, ans)

### Exercise 109. Calls with no disposition

`ans` = the number of `completed` calls with no row in `lead_calls` (`LEFT JOIN ... WHERE lc.call_log_id IS NULL`).

In [ ]:
ans = None   # replace None with your answer

check(109, ans)

### Exercise 110. WHERE versus HAVING

`ans` = a sorted list of leads with at least 5 dispositioned calls, of which at least 2 are
`'Interested'` or `'Call Later'`.

In [ ]:
ans = None   # replace None with your answer

check(110, ans)

## Module 11. Capstones

Each capstone rebuilds a real report from this week. Work through it in the same order every time:

1. **Inputs:** which tables, which rows (print counts).
2. **Types:** dates, durations, IDs as text.
3. **Clean:** filters and duplicates, counting what each step removed.
4. **Match:** joins with a before/after row count.
5. **Calculate:** helper columns first, then group.
6. **Check:** an `assert` for anything impossible, and a cross-check between two outputs.
7. **Output:** clear column names, then save.

Write each capstone from scratch, without scrolling up. If you can, you can do the real task without help.

### Capstones: exercises

### Exercise 111. The daily summary

From `D['calls']`, per `CRM ID` and `Date`: `Total_Dialed_Calls`, `Total_Connected_Calls`,
`Total_Talk_Time` (connected, whole seconds), `Total_Gap_Duration` (after the 1-hour allowance) and
`Login_Seconds` (exercise 61). `ans` = a DataFrame with those 7 columns.

In [ ]:
ans = None   # replace None with your answer

check(111, ans)

### Exercise 112. Hourly report with a built-in check

Build calls per slot per `CRM ID` and `Date` (exercise 64) plus a `Total Calls` column, and `assert` it equals
`Total_Dialed_Calls` from exercise 111. `ans` = the hourly DataFrame.

In [ ]:
ans = None   # replace None with your answer

check(112, ans)

### Exercise 113. Client activity report

For the stripped ids in `D['traded_ids']` (excluding `'No New Id'`) that have BILL rows: month columns of
trading days, `Total Trading Days`, `Average` (2 decimals), latest `Balance`, and `Last Trade` (latest BILL
`CreatedAt`). `ans` = a DataFrame with `ClientID` first.

In [ ]:
ans = None   # replace None with your answer

check(113, ans)

### Exercise 114. Status journey in pandas

Without SQL: keep one row per (`customer_id`, `occurred_at`) with the latest `created_at`, then per customer
`rt_status` (latest `new_status`) and `journey` (oldest first, `' -> '`).
`ans` = a DataFrame `customer_id`, `rt_status`, `journey`.

In [ ]:
ans = None   # replace None with your answer

check(114, ans)

### Exercise 115. Unreachable leads in pandas

Without SQL: join `call_logs`, `lead_calls` and `lead_call_dispositions`, skip placing/call_failed, rank each
lead's calls newest first (`groupby().cumcount()`), and `ans` = a sorted list of leads with at least 5 calls whose
last 5 are all in `D['UNREACHABLE']` (compare with `norm()`).

In [ ]:
ans = None   # replace None with your answer

check(115, ans)

## Your score

In [ ]:
progress()